# Table A — SSCD and CLIP Retrieval Ablation

Generates top-10 candidate retrieval manifests over the full image corpus for two baseline models:

| Model | What it is | Output dim |
|-------|-----------|------------|
| **SSCD** | Self-Supervised Copy Detection (Pizzi et al., CVPR 2022). Trained on DISC21 with copy-appropriate augmentations (JPEG, perspective warp, scan noise). The task-specific baseline. | 512 |
| **CLIP** | OpenAI ViT-L/14 image encoder. General semantic similarity — tests whether copy detection requires a task-specific model. | 768 |

No sharding is done here. The full manifests are written for all source images;
shard splitting happens in the downstream geometry filter step.

---

**Before running this notebook**, upload these files to Drive at `[DRIVE_ROOT]/scripts/`:

```
scripts/
    retrieve.py
    _local/
        ModelComboSSCD.py
        ModelComboCLIP.py
```

**Outputs** (written to Drive at the end):
```
ablation_outputs/retrieval/
    sscd_retrieval_manifest.jsonl   — top-10 per source, SSCD cosine scores
    sscd_feature_cache.npz          — source + target embeddings (512-dim)
    clip_retrieval_manifest.jsonl   — top-10 per source, CLIP cosine scores
    clip_feature_cache.npz          — source + target embeddings (768-dim)
```

In [1]:
# Step 0 — Install dependencies and verify GPU
!pip install timm open_clip_torch -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — switch runtime to GPU before continuing.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.6 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
CUDA available: True
GPU:   NVIDIA A100-SXM4-40GB
VRAM:  42.4 GB


In [2]:
# Step 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Step 2 — Configure paths
# ── EDIT THESE ──────────────────────────────────────────────────────────────
from pathlib import Path

DRIVE_ROOT  = Path('/content/drive/MyDrive/ImageSimilarity')
SCRIPTS_DIR = DRIVE_ROOT / 'scripts'            # contains retrieve.py + _local/
SOURCE_ZIP  = DRIVE_ROOT / 'workingsourcecrops.zip' # all source images (full corpus)
TARGET_ZIP  = DRIVE_ROOT / 'working_targetcrops.zip' # all target images (full corpus)
DRIVE_OUT   = DRIVE_ROOT / 'ablation_outputs' / 'retrieval'
TOP_K       = 10                                # candidates returned per source
# ────────────────────────────────────────────────────────────────────────────

import shutil, sys

LOCAL_WORK = Path('/content/retrieval_ablation')
SSCD_OUT   = LOCAL_WORK / 'sscd_output'
CLIP_OUT   = LOCAL_WORK / 'clip_output'

for d in [LOCAL_WORK, LOCAL_WORK / '_local', SSCD_OUT, CLIP_OUT, DRIVE_OUT]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Source dir : {SOURCE_ZIP}  (exists: {SOURCE_ZIP.exists()})")
print(f"Target dir : {TARGET_ZIP}  (exists: {TARGET_ZIP.exists()})")
print(f"Scripts dir: {SCRIPTS_DIR} (exists: {SCRIPTS_DIR.exists()})")
print(f"Drive out  : {DRIVE_OUT}")

Source dir : /content/drive/MyDrive/ImageSimilarity/workingsourcecrops.zip  (exists: True)
Target dir : /content/drive/MyDrive/ImageSimilarity/working_targetcrops.zip  (exists: True)
Scripts dir: /content/drive/MyDrive/ImageSimilarity/scripts (exists: True)
Drive out  : /content/drive/MyDrive/ImageSimilarity/ablation_outputs/retrieval


In [7]:
# Step 3 — Copy scripts from Drive to Colab local filesystem
to_copy = [
    (SCRIPTS_DIR / 'retrieve.py',                  LOCAL_WORK / 'retrieve.py'),
    (SCRIPTS_DIR / '_local' / 'ModelComboSSCD.py', LOCAL_WORK / '_local' / 'ModelComboSSCD.py'),
    (SCRIPTS_DIR / '_local' / 'ModelComboCLIP.py', LOCAL_WORK / '_local' / 'ModelComboCLIP.py'),
]

for src, dst in to_copy:
    if not src.exists():
        raise FileNotFoundError(
            f"Missing on Drive: {src}\nUpload it to {src.parent} and re-run this cell."
        )
    shutil.copy(src, dst)
    print(f"Copied: {src.name}")

sys.path.insert(0, str(LOCAL_WORK))
print("\nAll scripts ready.")

Copied: retrieve.py
Copied: ModelComboSSCD.py
Copied: ModelComboCLIP.py

All scripts ready.


In [8]:
# Step 4 - Copy and unzip corpus locally
import os

zips = [
    (SOURCE_ZIP, 'source.zip'),
    (TARGET_ZIP, 'target.zip'),
]

for src, dst in zips:
    if not src.exists():
        raise FileNotFoundError(
            f"Missing on Drive: {src}\nUpload it to {src.parent} and re-run this cell."
        )
    shutil.copy(src, dst)
    print(f"Copied: {src.name}")

!unzip target.zip -d target
!unzip source.zip -d source

src = "./target"

for root, dirs, files in os.walk(src):
    if root == src:
        continue  # skip top-level, files there are already flat
    for file in files:
        src_path = os.path.join(root, file)
        dst_path = os.path.join(src, file)

        # Handle filename collisions
        if os.path.exists(dst_path):
            base, ext = os.path.splitext(file)
            dst_path = os.path.join(src, f"{base}_{root.replace('/', '_')}{ext}")

        shutil.move(src_path, dst_path)

# Remove now-empty subdirectories
for root, dirs, files in os.walk(src, topdown=False):
    if root == src:
        continue
    if not os.listdir(root):
        os.rmdir(root)

print("Flattened Target Dir")

!cp ./target/* -r ./source
print("Copied target into source")

Streaming output truncated to the last 5000 lines.
  inflating: source/workingsourcecrops/3805-039648-0.jpg  
  inflating: source/workingsourcecrops/3805-039649-0.jpg  
  inflating: source/workingsourcecrops/3805-039650-1.jpg  
  inflating: source/workingsourcecrops/3805-039650-2.jpg  
  inflating: source/workingsourcecrops/3805-039651-0.jpg  
  inflating: source/workingsourcecrops/3805-039652-0_(1).jpg  
  inflating: source/workingsourcecrops/3805-039652-0_(2).jpg  
  inflating: source/workingsourcecrops/3805-039653-0.jpg  
  inflating: source/workingsourcecrops/3805-039654-0.jpg  
  inflating: source/workingsourcecrops/3805-039655-0.jpg  
  inflating: source/workingsourcecrops/3805-039656-0.jpg  
  inflating: source/workingsourcecrops/3805-039658-0.jpg  
  inflating: source/workingsourcecrops/3805-039659-0.jpg  
  inflating: source/workingsourcecrops/3805-039660-0.jpg  
  inflating: source/workingsourcecrops/3805-039661-0.jpg  
  inflating: source/workingsourcecrops/3805-039662-0.jpg

In [9]:
SOURCE_DIR = Path('./source')
TARGET_DIR = Path('./target')

In [12]:
src = "./source"

for root, dirs, files in os.walk(src):
    if root == src:
        continue  # skip top-level, files there are already flat
    for file in files:
        src_path = os.path.join(root, file)
        dst_path = os.path.join(src, file)

        # Handle filename collisions
        if os.path.exists(dst_path):
            base, ext = os.path.splitext(file)
            dst_path = os.path.join(src, f"{base}_{root.replace('/', '_')}{ext}")

        shutil.move(src_path, dst_path)

# Remove now-empty subdirectories
for root, dirs, files in os.walk(src, topdown=False):
    if root == src:
        continue
    if not os.listdir(root):
        os.rmdir(root)

print("Flattened Target Dir")

!cp ./target/* -r ./source
print("Copied target into source")

Flattened Target Dir
Copied target into source


## Smoke test *(recommended — run before the full retrieval)*

Loads both models and runs a forward pass on one image to catch import or
checkpoint errors early. Takes ~30 s. **Uncomment and run before proceeding.**

In [10]:

import importlib.util
import torch
from PIL import Image

def _load_model(path):
    spec = importlib.util.spec_from_file_location(path.stem, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return getattr(mod, path.stem)()

test_img_path = next(SOURCE_DIR.rglob('*.jpg'), next(SOURCE_DIR.rglob('*.png'), None))
if test_img_path is None:
    raise RuntimeError(f"No images found in {SOURCE_DIR} — check SOURCE_DIR path")
test_img = Image.open(test_img_path).convert('RGB')
print(f"Test image: {test_img_path.name}  size={test_img.size}")

for model_file in ['ModelComboSSCD.py', 'ModelComboCLIP.py']:
    model = _load_model(LOCAL_WORK / '_local' / model_file)
    model.eval()
    tensor = model.get_transform()(test_img).unsqueeze(0)
    with torch.no_grad():
        emb = model(tensor)
    print(f"{model_file}: output shape {tuple(emb.shape)}  ✓")
print("\nSmoke test passed.")

Test image: North China Magazine 1942-1-12_Page_323.jpg  size=(1402, 967)
Download complete.
ModelComboSSCD.py: output shape (1, 512)  ✓


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


ModelComboCLIP.py: output shape (1, 768)  ✓

Smoke test passed.


## Step 4 — SSCD retrieval

Featurizes all source and target images with SSCD (ResNet-50, 512-dim) then
returns the top-`TOP_K` candidates per source by cosine similarity.

Expected wall time on T4: **5–8 min** (~47 K targets × 2 ms/img + sources).

In [ ]:
import importlib
import retrieve
importlib.reload(retrieve)

retrieve.main([
    '--model-definition',   str(LOCAL_WORK / '_local' / 'ModelComboSSCD.py'),
    '--weights',            'none',
    '--source',             str(SOURCE_DIR),
    '--target',             str(TARGET_DIR),
    '--output-dir',         str(SSCD_OUT),
    '--manifest-name',      'sscd_retrieval_manifest.jsonl',
    '--feature-cache-name', 'sscd_feature_cache.npz',
    '--topx',               str(TOP_K),
    '--device',             'auto',
])

print(f"\nSSCD retrieval complete. Output dir: {SSCD_OUT}")

Found 42773 source images and 2856 target images
Loading model on cuda: /content/retrieval_ablation/_local/ModelComboSSCD.py
Using model-provided transform (RGB, no grayscale)
Featurizing source images
  featurized 100/42773
  featurized 200/42773
  featurized 300/42773
  featurized 400/42773
  featurized 500/42773
  featurized 600/42773
  featurized 700/42773
  featurized 800/42773
  featurized 900/42773
  featurized 1000/42773
  featurized 1100/42773
  featurized 1200/42773
  featurized 1300/42773
  featurized 1400/42773
  featurized 1500/42773
  featurized 1600/42773
  featurized 1700/42773
  featurized 1800/42773
  featurized 1900/42773
  featurized 2000/42773
  featurized 2100/42773
  featurized 2200/42773
  featurized 2300/42773
  featurized 2400/42773
  featurized 2500/42773
  featurized 2600/42773
  featurized 2700/42773
  featurized 2800/42773
  featurized 2900/42773
  featurized 3000/42773
  featurized 3100/42773
  featurized 3200/42773
  featurized 3300/42773
  featurized 34

In [ ]:
!zip -r SSCD_retrival.zip /content/retrieval_ablation/sscd_output

# 4. Copy the zipped file to your Google Drive
!cp SSCD_retrival.zip ./drive/MyDrive/ImageSimilarity/Colab_Transport

  adding: content/retrieval_ablation/sscd_output/ (stored 0%)
  adding: content/retrieval_ablation/sscd_output/sscd_feature_cache.npz (deflated 0%)
  adding: content/retrieval_ablation/sscd_output/sscd_retrieval_manifest.jsonl (deflated 94%)


## Step 5 — CLIP retrieval

Featurizes with CLIP ViT-L/14 (768-dim).  
Expected wall time on T4: **8–12 min** (ViT-L/14 is heavier than SSCD's ResNet-50).

In [ ]:
import importlib
import retrieve

importlib.reload(retrieve)

retrieve.main([
    '--model-definition',   str(LOCAL_WORK / '_local' / 'ModelComboCLIP.py'),
    '--weights',            'none',
    '--source',             str(SOURCE_DIR),
    '--target',             str(TARGET_DIR),
    '--output-dir',         str(CLIP_OUT),
    '--manifest-name',      'clip_retrieval_manifest.jsonl',
    '--feature-cache-name', 'clip_feature_cache.npz',
    '--topx',               str(TOP_K),
    '--device',             'auto',
])

print(f"\nCLIP retrieval complete. Output dir: {CLIP_OUT}")

Found 42773 source images and 2856 target images
Loading model on cuda: /content/retrieval_ablation/_local/ModelComboCLIP.py
Using model-provided transform (RGB, no grayscale)
Featurizing source images
  featurized 100/42773
  featurized 200/42773
  featurized 300/42773
  featurized 400/42773
  featurized 500/42773
  featurized 600/42773
  featurized 700/42773
  featurized 800/42773
  featurized 900/42773
  featurized 1000/42773
  featurized 1100/42773
  featurized 1200/42773
  featurized 1300/42773
  featurized 1400/42773
  featurized 1500/42773
  featurized 1600/42773
  featurized 1700/42773
  featurized 1800/42773
  featurized 1900/42773
  featurized 2000/42773
  featurized 2100/42773
  featurized 2200/42773
  featurized 2300/42773
  featurized 2400/42773
  featurized 2500/42773
  featurized 2600/42773
  featurized 2700/42773
  featurized 2800/42773
  featurized 2900/42773
  featurized 3000/42773
  featurized 3100/42773
  featurized 3200/42773
  featurized 3300/42773
  featurized 34

In [ ]:
!zip -r CLIP_retrival.zip /content/retrieval_ablation/clip_output

# 4. Copy the zipped file to your Google Drive
!cp clip_retrival.zip ./drive/MyDrive/ImageSimilarity/Colab_Transport

  adding: content/retrieval_ablation/clip_output/ (stored 0%)
  adding: content/retrieval_ablation/clip_output/clip_feature_cache.npz (deflated 0%)
  adding: content/retrieval_ablation/clip_output/clip_retrieval_manifest.jsonl (deflated 95%)
cp: cannot stat 'clip_retrival.zip': No such file or directory


## Step 6 — Copy outputs to Drive

In [ ]:
files_to_save = [
    SSCD_OUT / 'sscd_retrieval_manifest.jsonl',
    SSCD_OUT / 'sscd_feature_cache.npz',
    CLIP_OUT / 'clip_retrieval_manifest.jsonl',
    CLIP_OUT / 'clip_feature_cache.npz',
]

for src in files_to_save:
    if src.exists():
        dst = DRIVE_OUT / src.name
        shutil.copy(src, dst)
        print(f"Saved: {src.name}  ({src.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"MISSING: {src.name} — check for errors in the retrieval cells above")

print(f"\nAll outputs at: {DRIVE_OUT}")

## Step 7 — Verify outputs

Checks row counts and prints a sample row from each manifest.

In [ ]:
import json

for model_tag, manifest_path in [
    ('SSCD', SSCD_OUT / 'sscd_retrieval_manifest.jsonl'),
    ('CLIP', CLIP_OUT / 'clip_retrieval_manifest.jsonl'),
]:
    if not manifest_path.exists():
        print(f"{model_tag}: manifest not found — retrieval likely failed\n")
        continue

    lines = manifest_path.read_text(encoding='utf-8').strip().splitlines()
    rows = [json.loads(l) for l in lines]
    n_sources = len({r['source_id'] for r in rows})
    n_targets = len({r['target_id'] for r in rows})
    scores    = [r['similarity_score'] for r in rows]

    print(f"── {model_tag} {'─' * 40}")
    print(f"  Total rows : {len(rows):,}  (expected {n_sources} sources × {TOP_K} = {n_sources * TOP_K:,})")
    print(f"  Sources    : {n_sources}")
    print(f"  Unique tgts: {n_targets}")
    print(f"  Score range: {min(scores):.4f} – {max(scores):.4f}")
    r0 = rows[0]
    print(f"  Sample     : {r0['source_id']} → {r0['target_id']}  "
          f"rank={r0['rank']}  score={r0['similarity_score']:.4f}")
    print()

── SSCD ────────────────────────────────────────
  Total rows : 427,730  (expected 42773 sources × 10 = 427,730)
  Sources    : 42773
  Unique tgts: 2856
  Score range: 0.0945 – 0.9840
  Sample     : 3601-000003-1_(1) → North China Magazine 1942-1-12_Page_290_(5)  rank=1  score=0.2573

